# Main figures

Inputs:
- `input/co2block_basin_resources_closed.csv`: CO2BLOCK resource (Gt) and rate (Mt/yr) per basin and injection period, closed boundary (Figs 1, 3)
- `input/basin_map/global_sedimentary_basins.shp`: global sedimentary basin map polygons (Fig. 1)
- `input/basin_map/ne_50m_admin_0_countries.zip`: Natural Earth country outlines (Fig. 1)
- `output/central_pathways.csv`: central design pathway per scenario, jurisdiction and growth form (Figs 2, 3)
- `output/mc_samples.csv.gz`: accepted Monte Carlo draws of growth rate and storage resource (Fig. 3)
- `output/closed/screened_paths.csv`: annual demand and allocated injection rate for every allocation test (Fig. 2)
- `output/closed/screening_summary.csv`: allocation verdict for every test (Fig. 3)
- `output/closed/reconstruction_summary.csv`: original and reconstructed pathway parameters for failed tests (Figs 3, 5)
- `output/closed/basin_allocation.csv`: basin allocation steps for every test (Fig. 5)
- `output/closed/reconstruction_paths.csv`: annual original and reconstructed demand curves (Fig. 5)
- `output/closed/global_distribution_draws.csv.gz`: aggregate 2050 injection-rate draws before and after resource filtering (Fig. 4A)
- `output/closed/aggregation_coverage.csv`: jurisdictions included in each aggregate (Fig. 4B note)
- `output/closed/jurisdiction_rate2050_summary.csv`: jurisdiction 2050 rate quantiles before and after filtering (Fig. 4B)
- `output/ar6_world_rate2050.csv`: IPCC AR6 World CCS rates in 2050 by climate category (Fig. 4A)
- `output/ar6_regional_rate2050.csv`: IPCC AR6 regional CCS rates in 2050 mapped to study jurisdictions (Fig. 4B)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import FuncFormatter, MaxNLocator, NullLocator, StrMethodFormatter
from pyproj import Transformer
from scipy.special import logsumexp
from scipy.stats import gaussian_kde
from shapely.geometry import Polygon, box

ROOT = Path.cwd().resolve()
if not (ROOT / 'input').is_dir() and (ROOT.parent / 'input').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'input').is_dir(), 'Run from the Global Study folder or its code folder.'
INPUT, OUTPUT = ROOT / 'input', ROOT / 'output'
OUT = OUTPUT / 'closed'
FIGURES = OUTPUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

KEYS = ['scenario', 'country', 'model']
MODELS = ['Logistic', 'Gompertz']
SCENARIOS = ['minimum', 'growth10', 'policy', 'us1gt', 'reference', 'maximum']
SCEN_LABEL = dict(minimum='Minimum', growth10='Growth 10%', policy='Policy', us1gt='US 1 Gt',
                  reference='Reference', maximum='Maximum')
SCEN_COLOR = dict(minimum='#777777', growth10='#B29B42', policy='#449A87', us1gt='#447733',
                  reference='#443388', maximum='#BB5577')
BASE_RC = {'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.titlesize': 8, 'axes.labelsize': 7,
           'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'legend.fontsize': 6.5, 'figure.titlesize': 9,
           'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6,
           'lines.linewidth': .9, 'pdf.fonttype': 42, 'ps.fonttype': 42}

def read_csv(path, **kw):
    return pd.read_csv(path, float_precision='round_trip', **kw)

def save(fig, name, dpi=300):
    for ext in ['pdf', 'png']:
        fig.savefig(FIGURES / f'{name}.{ext}', dpi=dpi, facecolor='white')
    plt.close(fig)

central = read_csv(OUTPUT / 'central_pathways.csv')
screening = read_csv(OUT / 'screening_summary.csv')
reconstruction = read_csv(OUT / 'reconstruction_summary.csv')
resources = read_csv(INPUT / 'co2block_basin_resources_closed.csv')
resources = resources[resources.InStudyScope]
CAPACITY_150 = resources[resources.Period_yr.eq(150)].groupby('ScreeningCase').Max_Capacity_Gt.sum()

## Figure 1. Basin resources

In [ ]:
EDGES = np.array([1e-3, 1e-2, 1e-1, 1, 10, 100, 1000, 1e4])
PALETTE = ['#f0f9e8', '#ccebc5', '#a8ddb5', '#7bccc4', '#4eb3d3', '#2b8cbe', '#08589e']
RANK_COLOR = {1: '#f03b20', 2: '#feb24c', 3: '#fa9fb5'}
INK, MUTED = '#24383F', '#687A80'
NAMES = {'Canada': 'Canada', 'UK': 'United Kingdom', 'EU': 'EU group', 'Middle East': 'Middle East',
         'Brazil': 'Brazil', 'Thailand': 'Thailand', 'Indonesia': 'Indonesia', 'Australia': 'Australia',
         'US': 'United States', 'China': 'China'}
MAP_LABELS = {'Canada': ((-150, 43), (-106, 60)), 'US': ((-142, 28), (-112, 39)),
              'Brazil': ((-31, -27), (-47, -15)), 'UK': ((-30, 48), (-2, 54)), 'EU': ((58, 60), (15, 54)),
              'China': ((90, 32), None), 'Middle East': ((67, -11), (48, 25)),
              'Thailand': ((74, 0), (101, 14)), 'Indonesia': ((96, -23), (110, -6)),
              'Australia': ((130, -45), None)}
colour_of = lambda value: PALETTE[int(np.searchsorted(EDGES, value, side='right') - 1)]
log_pos = lambda value, lo, hi: (np.log10(value) - np.log10(lo)) / (np.log10(hi) - np.log10(lo))
short = lambda value: f'{value / 1000:g}k' if value >= 1000 else f'{value:g}'

b150 = resources[resources.Period_yr.eq(150)].assign(Region_name=lambda d: d.Region_name.str.strip())
geo = gpd.read_file(INPUT / 'basin_map/global_sedimentary_basins.shp')
geo['Region_name'] = geo['Basin Name'].str.strip()
geo = geo.loc[geo.Region_name.isin(b150.Region_name), ['Region_name', 'geometry']]
study = geo.merge(b150, on='Region_name', validate='one_to_one')
assert len(study) == 131
study['colour'] = study.Max_Capacity_Gt.map(colour_of)
ranked = study.sort_values(['Max_Capacity_Gt', 'Region_no'], ascending=[False, True])
study['rank'] = pd.Series(np.arange(1, len(study) + 1), index=ranked.index)
cuts = [int(np.floor(len(study) * p / 100)) for p in (10, 20, 30)]
study['band'] = np.select([study['rank'] <= c for c in cuts], [1, 2, 3], 0)
shares = [ranked.Max_Capacity_Gt.cumsum().iloc[c - 1] / study.Max_Capacity_Gt.sum() * 100 for c in cuts]
totals = resources[resources.Period_yr.isin([20, 70, 150])].groupby(['ScreeningCase', 'Period_yr']).Max_Capacity_Gt.sum()

ROBINSON = '+proj=robin +lon_0=0 +datum=WGS84 +units=m +no_defs'
project = Transformer.from_crs('EPSG:4326', ROBINSON, always_xy=True)
countries = gpd.read_file(INPUT / 'basin_map/ne_50m_admin_0_countries.zip')
countries = countries[countries.CONTINENT != 'Antarctica'].to_crs('EPSG:4326')
countries.geometry = countries.geometry.make_valid().intersection(box(-180, -58, 180, 85))
countries = countries[~countries.geometry.is_empty].to_crs(ROBINSON)
lons, lats = np.linspace(-180, 180, 721), np.linspace(-58, 85, 361)
ring = ([(x, -58) for x in lons] + [(180, y) for y in lats] + [(x, 85) for x in lons[::-1]]
        + [(-180, y) for y in lats[::-1]])
world = gpd.GeoSeries([Polygon(ring)], crs='EPSG:4326').to_crs(ROBINSON)
study_r = study.to_crs(ROBINSON)

In [ ]:
def basin_resource_figure():
    W, WIN_W, WIN_H = 183.0, 30.0, 40.0
    map_left = 3.0 + WIN_W + 2.5
    map_w = W - 2 * map_left
    xmin, ymin, xmax, ymax = world.total_bounds
    map_h = (ymax - ymin) * map_w / (xmax - xmin)
    key_y0, key_y1 = 3.0, 18.0
    bot_y = key_y1 + 2.5
    map_bottom = bot_y + WIN_H + 1.5
    top_y = map_bottom + map_h + 2.5
    H = round(top_y + WIN_H + 2.0, 1)
    cols = [3.0 + i * (WIN_W + (W - 6.0 - 4 * WIN_W) / 3) for i in range(4)]
    side_y = map_bottom + (map_h - WIN_H) / 2
    windows = {**{k: (cols[i], top_y) for i, k in enumerate(['Canada', 'UK', 'EU', 'Middle East'])},
               **{k: (cols[i], bot_y) for i, k in enumerate(['Brazil', 'Thailand', 'Indonesia', 'Australia'])},
               'US': (3.0, side_y), 'China': (W - 3.0 - WIN_W, side_y)}

    fig = plt.figure(figsize=(W / 25.4, H / 25.4))
    def line(x0, y0, x1, y1, **kw):
        fig.add_artist(Line2D([x0 / W, x1 / W], [y0 / H, y1 / H], transform=fig.transFigure, **kw))
    def point(x, y, **kw):
        fig.add_artist(Line2D([x / W], [y / H], linestyle='none', transform=fig.transFigure, **kw))
    def rect(x, y, w, h, **kw):
        fig.add_artist(Rectangle((x / W, y / H), w / W, h / H, transform=fig.transFigure, **kw))
    def text(x, y, s, **kw):
        fig.text(x / W, y / H, s, **kw)

    ax = fig.add_axes([map_left / W, map_bottom / H, map_w / W, map_h / H])
    world.plot(ax=ax, facecolor='#FFFFFF', edgecolor='none', zorder=0)
    countries.plot(ax=ax, facecolor='#F0F1EE', edgecolor='#BAC4C2', linewidth=.25, zorder=1)
    study_r.plot(ax=ax, color=study_r.colour, edgecolor='#45616A', linewidth=.30, zorder=3)
    for band, colour in RANK_COLOR.items():
        edges = study_r[study_r.band.eq(band)].boundary
        edges.plot(ax=ax, color='white', linewidth=1.35, zorder=4)
        edges.plot(ax=ax, color='#47515B', linewidth=1.10, zorder=5)
        edges.plot(ax=ax, color=colour, linewidth=.80, zorder=6)
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
    ax.set_aspect('equal')
    ax.set_axis_off()
    for key, (label_at, anchor) in MAP_LABELS.items():
        name = {'UK': 'UK'}.get(key, NAMES[key])
        if anchor is None:
            ax.text(*project.transform(*label_at), name, fontsize=6, color=INK, ha='center', va='center', zorder=7)
        else:
            ax.annotate(name, project.transform(*anchor), xytext=project.transform(*label_at), ha='center',
                        va='center', fontsize=6, color=INK, zorder=7,
                        arrowprops={'arrowstyle': '-', 'color': '#82959A', 'lw': .35, 'shrinkA': 3, 'shrinkB': 3})

    for key, (left, bottom) in windows.items():
        at = lambda u, v: (left + WIN_W * u, bottom + WIN_H * v)
        sub = study[study.ScreeningCase.eq(key)]
        title_u = .21 if key == 'EU' else .035
        text(*at(title_u, .970), NAMES[key], fontsize=8, weight='bold', va='center', color=INK)
        text(*at(.975, .880), f'n={len(sub)}', fontsize=6, va='center', ha='right', color=INK)
        text(*at(.035, .880), 'Resource (Gt CO₂)', fontsize=6, va='center', color=MUTED)
        text(*at(.035, .455), '150-year rate (Mt yr⁻¹)', fontsize=6, va='center', color=MUTED)
        for ticks, (lo, hi), (v0, v1), grid in [((1e1, 1e2, 1e3, 1e4), (1e1, 1e4), (.590, .815), False),
                                                ((1e-2, 1e0, 1e2, 1e4), (1e-2, 1e5), (.075, .395), True)]:
            line(*at(.215, v0), *at(.215, v1), color='#BDC8CE', lw=.42)
            line(*at(.215, v0), *at(.975, v0), color='#BDC8CE', lw=.42)
            for tick in ticks:
                v = v0 + log_pos(tick, lo, hi) * (v1 - v0)
                if grid:
                    line(*at(.215, v), *at(.975, v), color='#E6EBED', lw=.32, zorder=0)
                line(*at(.198, v), *at(.215, v), color='#A7B5BD', lw=.38)
                text(*at(.184, v), short(tick), fontsize=6, ha='right', va='center', color=MUTED)
        for period, u, marker in zip((20, 70, 150), (.35, .625, .90), 'oDs'):
            value = totals[key, period]
            v = .590 + log_pos(value, 1e1, 1e4) * (.815 - .590)
            point(*at(u, v), marker=marker, markersize=3.7, markerfacecolor=colour_of(value),
                  markeredgecolor='#384757', markeredgewidth=.45, zorder=4)
            text(*at(u, v + .055), f'{value:.0f}' if value >= 100 else f'{value:.1f}',
                 fontsize=7, ha='center', va='center', color=INK)
            line(*at(u, .578), *at(u, .590), color='#A7B5BD', lw=.4)
            text(*at(u, .545), str(2030 + period), fontsize=6, ha='center', va='center', color=INK)
        ry = lambda rate: .075 + log_pos(rate, 1e-2, 1e5) * (.395 - .075)
        xc = (.215 + .975) / 2
        ordered = sub.sort_values(['Region_Q_Mt_yr', 'Region_no'])
        rates = ordered.Region_Q_Mt_yr
        q1, median, q3 = np.percentile(rates, [25, 50, 75])
        lower, upper = rates[rates >= q1 - 1.5 * (q3 - q1)].min(), rates[rates <= q3 + 1.5 * (q3 - q1)].max()
        line(*at(xc, ry(lower)), *at(xc, ry(upper)), color='#8A99A6', lw=.65, zorder=1)
        for end in (lower, upper):
            line(*at(xc - .04, ry(end)), *at(xc + .04, ry(end)), color='#8A99A6', lw=.55, zorder=1)
        rect(*at(xc - .09, ry(q1)), .18 * WIN_W, (ry(q3) - ry(q1)) * WIN_H,
             facecolor='#E0E3E8', edgecolor='#9CA9B6', linewidth=.50, zorder=1)
        line(*at(xc - .09, ry(median)), *at(xc + .09, ry(median)), color='#4C5664', lw=.90, zorder=2)
        w_pt, h_pt, sep = WIN_W / 25.4 * 72, WIN_H / 25.4 * 72, 2.8
        candidates = [0] + [s * k * sep / w_pt for k in range(1, 10) for s in (-1, 1)]
        placed = []
        for basin in ordered.itertuples():
            v = ry(basin.Region_Q_Mt_yr)
            dx = next(c for c in candidates if all(abs(c - px) * w_pt >= sep - .01 or abs(v - pv) * h_pt >= sep
                                                   for px, pv in placed))
            placed.append((dx, v))
            if basin.band:
                point(*at(xc + dx, v), marker='s', markersize=2.75, markerfacecolor='none',
                      markeredgecolor='#596573', markeredgewidth=.40, zorder=3)
            point(*at(xc + dx, v), marker='s', markersize=2.1, markerfacecolor=basin.colour,
                  markeredgecolor=RANK_COLOR.get(basin.band, '#596573'),
                  markeredgewidth=.85 if basin.band else .28, zorder=4)

    rect(3.0, key_y0, W - 6.0, key_y1 - key_y0, facecolor='white', edgecolor='black', linewidth=.65, zorder=0)
    text(7.0, key_y1 - 1.2, 'Resource (Gt CO₂)', fontsize=7, weight='bold', va='top', color=INK)
    bar_y = key_y0 + 5.4
    for i, colour in enumerate(PALETTE):
        rect(7.0 + 6.0 * i, bar_y, 6.0, 2.4, facecolor=colour, edgecolor='white', linewidth=.25)
    for i, edge in enumerate(EDGES):
        line(7.0 + 6.0 * i, bar_y - .7, 7.0 + 6.0 * i, bar_y, color='#798095', lw=.30)
        text(7.0 + 6.0 * i, bar_y - 2.2, short(edge), fontsize=6, ha='center', va='center', color=MUTED)
    cx, bar_w = 63.0, 100.0
    text(59.11, key_y1 - 1.2, 'Study-basin resource concentration (150 years)', fontsize=7, weight='bold',
         va='top', color=INK)
    bounds = [0.0, *shares, 100.0]
    labels = ['top 10% of basins', '10–20%', '20–30%', 'other 70%']
    for k, (c0, c1, colour, label) in enumerate(zip(bounds[:-1], bounds[1:], [*RANK_COLOR.values(), '#D9DEE3'], labels)):
        x0, x1 = cx + bar_w * c0 / 100, cx + bar_w * c1 / 100
        rect(x0, bar_y, x1 - x0, 2.4, facecolor=colour, edgecolor='#52606A', linewidth=.3)
        if k < 3:
            text((x0 + x1) / 2, bar_y + 4.0, label, fontsize=6, ha='center', va='center', color=INK)
        else:
            text(x1 + 1.2, bar_y + 1.2, label, fontsize=6, ha='left', va='center', color=INK)
        line(x1, bar_y - .6, x1, bar_y, color='#52606A', lw=.35)
        y_label = bar_y - (3.6 if k == 2 else 2.0)
        if k == 2:
            line(x1, bar_y - .6, x1, y_label + 1.6, color='#52606A', lw=.3)
        text(x1, y_label, f'{c1:.1f}%' if c1 < 100 else '100%', fontsize=6, ha='center', va='center', color=INK)
    text(cx, bar_y - 2.0, '0%', fontsize=6, ha='center', va='center', color=INK)
    return fig

with mpl.rc_context({'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans'], 'pdf.fonttype': 42}):
    save(basin_resource_figure(), 'fig1_basin_resources')

## Figure 2. Demand–supply timeline

In [ ]:
FIG2_RC = {'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
           'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.linewidth': .6,
           'xtick.major.width': .6, 'xtick.major.size': 2.5}
RATIO_CMAP = LinearSegmentedColormap.from_list(
    'ratio', [(0, '#FFFFFF'), (.5, '#2F4B7C'), (.5001, '#F4A582'), (1, '#67001F')])
to_z = lambda ratio: np.where(ratio <= 1, .5 * ratio, .5 + .5 * np.minimum(np.log2(np.maximum(ratio, 1)) / 4, 1))

def timeline_figure():
    paths = read_csv(OUT / 'screened_paths.csv', usecols=KEYS + ['allocation_order', 'year', 'demand_rate_mt_yr',
                                                                  'allocation_level_mt_yr'])
    paths['ratio'] = paths.demand_rate_mt_yr / paths.allocation_level_mt_yr
    ratio = {k: g.sort_values('year').ratio.to_numpy() for k, g in paths.groupby(KEYS + ['allocation_order'])}
    peak_year = central.set_index(KEYS).tp
    scenarios = ['maximum', 'reference', 'us1gt', 'policy', 'growth10', 'minimum']
    jurisdictions = ['China', 'US', 'Middle East', 'EU', 'Brazil', 'Canada', 'Indonesia', 'Australia', 'UK', 'Thailand']
    ink, light = '#272727', '#D9D9D9'
    fs = dict(jur=6.3, scen=7.0, title=7.0, group=7.5, letter=8.0, tick=7.0, cbar=6.8)
    W, H, TOP, PW, PH = 178.0, 170.0, 14.0, 33.5, 136.0
    nrow = len(scenarios) * len(jurisdictions)
    row_h = PH / nrow
    fig = plt.figure(figsize=(W / 25.4, H / 25.4))
    axes_mm = lambda x, top, w, h: fig.add_axes([x / W, 1 - (top + h) / H, w / W, h / H])
    def text(x, top, s, **kw):
        fig.text(x / W, 1 - top / H, s, color=ink, **kw)
    def line(x0, top0, x1, top1, **kw):
        fig.add_artist(Line2D([x0 / W, x1 / W], [1 - top0 / H, 1 - top1 / H], transform=fig.transFigure, **kw))
    def rect(x, top, w, h, **kw):
        fig.add_artist(Rectangle((x / W, 1 - (top + h) / H), w / W, h / H, transform=fig.transFigure, lw=0, **kw))

    for model, order, x0, letter in [('Logistic', 'ascend', 27.0, 'A'), ('Logistic', 'descend', 63.5, 'B'),
                                     ('Gompertz', 'ascend', 107.0, 'C'), ('Gompertz', 'descend', 143.5, 'D')]:
        ax = axes_mm(x0, TOP, PW, PH)
        rows = [(s, j) for s in scenarios for j in jurisdictions]
        grid = np.array([to_z(ratio[s, j, model, order]) for s, j in rows])
        im = ax.imshow(grid, aspect='auto', cmap=RATIO_CMAP, norm=mpl.colors.Normalize(0, 1),
                       interpolation='nearest', extent=[2029.5, 2180.5, nrow - .5, -.5])
        for i, (s, j) in enumerate(rows):
            t = peak_year[s, j, model]
            if t <= 2180.5:
                ax.plot([t, t], [i - .45, i + .45], color='white', lw=1.2, solid_capstyle='butt', zorder=3)
        for k in range(1, len(scenarios)):
            ax.axhline(k * len(jurisdictions) - .5, color='white', lw=1.4)
        ax.axvline(2050, color=ink, lw=.6, ls=(0, (2.5, 1.5)), zorder=4)
        ax.set_xlim(2029.5, 2180.5)
        ax.set_xticks([2050, 2100, 2150])
        ax.tick_params(axis='x', labelsize=fs['tick'], pad=1.5, length=2.5)
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        text(x0 - 1.2, 10.6, letter, fontsize=fs['letter'], fontweight='bold', va='baseline', ha='left')
        text(x0 + 2.6, 10.6, dict(ascend='Ascending order', descend='Descending order')[order],
             fontsize=fs['title'], va='baseline', ha='left')
    for name, x0 in [('Logistic growth', 27.0), ('Gompertz growth', 107.0)]:
        text(x0, 4.6, name, fontsize=fs['group'], fontweight='bold', va='baseline')
        line(x0, 6.4, x0 + 2 * PW + 3.0, 6.4, color=light, lw=.6)
    for k, s in enumerate(scenarios):
        y_top = TOP + k * len(jurisdictions) * row_h
        rect(8.6, y_top + .3, 1.4, len(jurisdictions) * row_h - .6, color=SCEN_COLOR[s])
        text(5.4, y_top + len(jurisdictions) * row_h / 2, SCEN_LABEL[s], rotation=90, fontsize=fs['scen'],
             ha='center', va='center')
        for i, j in enumerate(jurisdictions):
            text(26.0, y_top + (i + .5) * row_h, 'EU group' if j == 'EU' else j, fontsize=fs['jur'],
                 ha='right', va='center')

    cb_top = TOP + PH + 9.0
    cb = fig.colorbar(im, cax=axes_mm(52.0, cb_top, 64.0, 2.4), orientation='horizontal',
                      ticks=[0, .25, .5, .625, .75, 1])
    cb.ax.set_xticklabels(['0', '0.5', '1', '2', '4', '≥16'])
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=fs['cbar'], length=2, pad=1.2)
    cb.set_label('Annual demand / allocated injection rate', fontsize=fs['tick'], labelpad=2.5)
    y_key = cb_top + 1.2
    rect(122.0, y_key - 1.9, 5.2, 3.8, color='#2F4B7C')
    line(124.6, y_key - 1.7, 124.6, y_key + 1.7, color='white', lw=1.2)
    text(128.6, y_key, 'Demand peak', fontsize=fs['cbar'], va='center')
    line(148.5, y_key - 1.7, 148.5, y_key + 1.7, color=ink, lw=.6, ls=(0, (2.5, 1.5)))
    text(150.3, y_key, '2050', fontsize=fs['cbar'], va='center')
    return fig

with mpl.rc_context(FIG2_RC):
    save(timeline_figure(), 'fig2_demand_supply_timeline', dpi=400)

## Figure 3. Growth and storage resources

In [ ]:
samples = read_csv(OUTPUT / 'mc_samples.csv.gz', usecols=KEYS + ['g_draw', 'C_Gt'])

def growth_resource_figure():
    order = ['US', 'China', 'Canada', 'Australia', 'UK', 'Brazil', 'EU', 'Thailand', 'Indonesia', 'Middle East']
    form_color = {'Logistic': '#337DA5', 'Gompertz': '#CB8133'}
    asc = screening[screening.allocation_order.eq('ascend')]
    rec = reconstruction[reconstruction.allocation_order.eq('ascend')]
    unresolved = rec[~rec.landmark_solved.eq(True)]
    recon = rec[rec.landmark_solved.eq(True)].merge(asc.loc[~asc.allocation_pass, KEYS], on=KEYS)
    cloud = samples[samples.scenario.isin(['minimum', 'growth10', 'reference', 'maximum'])]
    goal = central.set_index(['country', 'scenario', 'model']).C_scenario_Gt
    ymin = 10. ** np.floor(np.log10(min(cloud.C_Gt.min(), central.C_scenario_Gt.min(), recon.C_recon_order_gt.min())))
    ymax = 10. ** np.ceil(np.log10(max(cloud.C_Gt.max(), central.C_scenario_Gt.max(), CAPACITY_150.max(),
                                       recon.C_recon_order_gt.max())))
    xmax = max(30., 5 * np.ceil(recon.g_recon_order.max() * 100 / 5))
    X, logY = np.meshgrid(np.linspace(0, xmax, 90), np.linspace(np.log10(ymin), np.log10(ymax), 90))
    Y = 10 ** logY

    fig, axes = plt.subplots(2, 5, figsize=(18.3 / 2.54, 9.0 / 2.54), sharex=True, sharey=True)
    fig.subplots_adjust(left=.062, right=.992, bottom=.265, top=.95, hspace=.42, wspace=.14)
    for ax, country in zip(axes.flat, order):
        for model in MODELS:
            d = cloud[cloud.country.eq(country) & cloud.model.eq(model)]
            density = gaussian_kde(np.vstack([d.g_draw.to_numpy() * 100, np.log10(d.C_Gt)]))(
                np.vstack([X.ravel(), logY.ravel()])).reshape(X.shape)
            density /= density.max()
            ax.contourf(X, Y, density, levels=[.12, .35, .65, .90, 1.01], colors=[form_color[model]], alpha=.16, zorder=1)
            ax.contour(X, Y, density, levels=[.12, .35, .65, .90], colors=[form_color[model]], linewidths=.35,
                       alpha=.6, zorder=2)
        ax.axhline(CAPACITY_150[country], color='#222222', linewidth=.8, zorder=4)
        g_cap = central.loc[central.country.eq(country) & central.scenario.eq('reference'), 'g_cap'].iloc[0] * 100
        ax.axvline(g_cap, color='#6B747A', linestyle=(0, (1.5, 2)), linewidth=.6, zorder=3)
        for scenario, tag in [('minimum', 'Min'), ('reference', 'Ref'), ('maximum', 'Max')]:
            c_goal = goal[country, scenario, 'Logistic']
            ax.axhline(c_goal, color='#68757C', linestyle=(0, (4, 2)), linewidth=.55, zorder=3)
            label = ax.text(.025, c_goal * 1.05, tag, transform=ax.get_yaxis_transform(), fontsize=6,
                            color='#3D484F', va='bottom', zorder=6)
            label.set_path_effects([pe.withStroke(linewidth=1.3, foreground='white')])
        for r in recon[recon.country.eq(country)].itertuples():
            x, y = r.g_recon_order * 100, r.C_recon_order_gt
            ax.annotate('', xy=(x, y), xytext=(x, goal[country, r.scenario, r.model]), zorder=4,
                        arrowprops=dict(arrowstyle='->', color=SCEN_COLOR[r.scenario], lw=.55, shrinkA=1, shrinkB=1))
            ax.scatter([x], [y], s=12, marker='o' if r.model == 'Logistic' else '^', color=SCEN_COLOR[r.scenario],
                       edgecolor='white', linewidth=.3, zorder=7)
            if not r.growth_cap_pass:
                ax.scatter([x], [y], s=31, marker='o', facecolor='none', edgecolor='#202A30', linewidth=.7, zorder=8)
        missing = unresolved[unresolved.country.eq(country)]
        if len(missing):
            note = '; '.join(SCEN_LABEL[s] + ': ' + ', '.join(g.model.str[0]) for s, g in missing.groupby('scenario', sort=False))
            ax.text(.02, .02, 'Reconstruction infeasible:\n' + note, transform=ax.transAxes, fontsize=6,
                    color='#8b3434', va='bottom', bbox=dict(facecolor='white', edgecolor='none', alpha=.9, pad=.5))
        ax.set_yscale('log')
        ax.set_xlim(0, xmax)
        ax.set_ylim(ymin, ymax * 1.4)
        ax.set_xticks([0, 10, 20, 30])
        yticks = [10. ** p for p in range(int(np.floor(np.log10(ymin))), int(np.ceil(np.log10(ymax))) + 1, 2)]
        ax.set_yticks(yticks, [f'{v:g}' for v in yticks])
        ax.minorticks_off()
        ax.set_title('EU group' if country == 'EU' else country, fontsize=7, pad=3)
        ax.grid(axis='y', color='#EBEEF0', linewidth=.3, zorder=0)
        ax.spines[['left', 'bottom']].set_linewidth(.5)
        ax.tick_params(length=2, width=.5, pad=1, labelsize=6)
    handles = [Patch(facecolor=form_color['Logistic'], alpha=.35, label='Logistic MC'),
               Patch(facecolor=form_color['Gompertz'], alpha=.35, label='Gompertz MC'),
               Line2D([], [], color='#222222', lw=.8, label='150-year basin resource'),
               Line2D([], [], marker='o', color='#444444', ls='none', markersize=3, label='Logistic reconstruction'),
               Line2D([], [], marker='^', color='#444444', ls='none', markersize=3, label='Gompertz reconstruction'),
               Line2D([], [], marker='o', color='#222222', markerfacecolor='none', ls='none', markersize=4,
                      label='g exceeds limit')]
    scenario_handles = [Line2D([], [], color=SCEN_COLOR[s], marker='o', ls='-', lw=.6, markersize=2.5, label=SCEN_LABEL[s])
                        for s in ['minimum', 'growth10', 'reference', 'maximum', 'us1gt', 'policy']]
    legend = fig.legend(handles=[h for pair in zip(handles, scenario_handles) for h in pair], loc='lower center',
                        bbox_to_anchor=(.524, .012), ncol=6, fontsize=6, frameon=True, fancybox=False,
                        edgecolor='black', framealpha=1., handlelength=1.3, handletextpad=.5, columnspacing=1.6,
                        labelspacing=.55, borderpad=.7)
    legend.get_frame().set_linewidth(.65)
    fig.text(.527, .165, 'Common growth rate g (%)', ha='center', fontsize=7)
    fig.text(.003, .61, 'Storage resource (Gt)', rotation=90, va='center', fontsize=7)
    return fig

with mpl.rc_context({**BASE_RC, 'axes.titlesize': 7, 'xtick.labelsize': 6, 'ytick.labelsize': 6}):
    save(growth_resource_figure(), 'fig3_growth_resources')

## Figure 4. 2050 injection rates against AR6

In [ ]:
RATE_COLOR = dict(minimum='#888888', growth10='#DDCC77', policy='#44AA99', us1gt='#117733',
                  reference='#332288', maximum='#CC6677')
AR6_COLOR = dict(C1='#08306b', C2='#2171b5', C3='#4292c6', C4='#9ecae1')
STAGES = ['pre', 'post']

draws = read_csv(OUT / 'global_distribution_draws.csv.gz', usecols=['scenario', 'model', 'pre_rate_2050', 'post_rate_2050'])
coverage = read_csv(OUT / 'aggregation_coverage.csv')
intervals = read_csv(OUT / 'jurisdiction_rate2050_summary.csv').set_index(KEYS + ['stage'])
ar6_world = read_csv(OUTPUT / 'ar6_world_rate2050.csv')
ar6_regional = read_csv(OUTPUT / 'ar6_regional_rate2050.csv')

rates = {(st, s, m): draws.loc[draws.scenario.eq(s) & draws.model.eq(m), f'{st}_rate_2050'].to_numpy(float)
         for s in SCENARIOS for m in MODELS for st in STAGES}
bins = {}
for s in SCENARIOS:
    v = np.concatenate([rates[st, s, m] for m in MODELS for st in STAGES])
    pad = max((v.max() - v.min()) * .04, 1e-3)
    bins[s] = np.linspace(max(0., v.min() - pad), v.max() + pad, 33)
ar6_values = {c: ar6_world.loc[ar6_world.category.eq(c), 'rate_gt_yr'].to_numpy(float) for c in AR6_COLOR}
XMAX = float(np.ceil(max(max(b[-1] for b in bins.values()), max(v.max() for v in ar6_values.values())) * 1.02))
YMAX = max(np.histogram(v, bins[s])[0].max() for (st, s, m), v in rates.items()) * 1.05
high = max(intervals.xs('pre', level='stage').maximum.max(), ar6_regional.rate_gt_yr.max())
LIMITS = (0., float(max(20., np.ceil(high / 10.) * 10.) * 10 ** .15))
partial = coverage[~coverage.coverage_complete]
COVERAGE_NOTE = ('; '.join(f'{r.scenario.capitalize()} {r.model[0]}: {r.n_jurisdictions} (excl. {r.excluded_jurisdictions})'
                           for r in partial.itertuples()) + '; other groups: 10. Before/after coverage matched.'
                 if len(partial) else 'All groups: 10 jurisdictions; before/after coverage matched.')

In [ ]:
def histogram_band(ax, stage):
    sign = 1 if stage == 'pre' else -1
    for s in SCENARIOS:
        for m in MODELS:
            counts, _ = np.histogram(rates[stage, s, m], bins[s])
            if m == 'Logistic':
                ax.stairs(sign * counts, bins[s], fill=True, color=RATE_COLOR[s], alpha=.30, lw=0)
            ax.stairs(sign * counts, bins[s], color=RATE_COLOR[s], ls='-' if m == 'Logistic' else (0, (3.2, 1.6)),
                      lw=1.0 if m == 'Logistic' else 1.35, alpha=.95, baseline=None, zorder=3 if m == 'Logistic' else 4)
    ax.set_xlim(0, XMAX)
    ax.set_ylim((0, YMAX) if stage == 'pre' else (-YMAX, 0))
    ticks = MaxNLocator(5).tick_values(0, YMAX)
    ticks = ticks[(ticks >= 0) & (ticks <= YMAX)]
    ax.set_yticks(sign * ticks)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{abs(v):.0f}'))
    ax.set_ylabel('Aggregate draws', fontsize=7)
    ax.grid(axis='y', color='#eeeeee', lw=.5)
    ax.set_axisbelow(True)
    if stage == 'pre':
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(axis='x', bottom=False, labelbottom=False)
    else:
        ax.set_xlabel('Global annual injection rate in 2050 (Gt yr⁻¹)', fontsize=7, labelpad=3)
    mid = np.mean(np.sort(np.abs(ax.get_yticks()))[-2:]) / YMAX
    ax.text(.985, mid if stage == 'pre' else 1 - mid, 'Before resource filtering' if stage == 'pre'
            else 'After resource filtering', transform=ax.transAxes, ha='right', va='center', fontsize=7, color='.3')

def density_band(ax, stage):
    x = np.linspace(.01, XMAX, 800)
    pooled = [np.concatenate([rates[stage, s, m] for m in MODELS]) for s in SCENARIOS]
    log_density = np.asarray([gaussian_kde(v, bw_method=.10).logpdf(x) for v in pooled])
    log_total = logsumexp(log_density, axis=0)
    lo, hi = min(b[0] for b in bins.values()), max(b[-1] for b in bins.values())
    track = (x >= lo) & (x <= hi)
    coloured = track & (x >= min(v.min() for v in pooled)) & (x <= max(v.max() for v in pooled))
    cumulative = np.cumsum(np.exp(log_density - log_total[None, :]), axis=0)
    yy = (np.arange(140) + .5) / 140
    index = np.clip((cumulative[None, :, :] < yy[:, None, None]).sum(axis=1), 0, 5)
    rgba = np.zeros((140, len(x), 4))
    rgba[..., :3] = np.asarray([mpl.colors.to_rgb(RATE_COLOR[s]) for s in SCENARIOS])[index]
    rgba[..., 3] = np.where(coloured, .15 + .85 * np.exp(.35 * (log_total - log_total[track].max())), 0.)[None, :]
    ax.add_patch(Rectangle((lo, 0), hi - lo, 1, facecolor='#e6e8eb', edgecolor='#bcc2c8', lw=.35, zorder=0))
    ax.imshow(rgba, extent=[x[0], x[-1], 0, 1], aspect='auto', origin='lower', interpolation='nearest', zorder=1)
    ax.set(xlim=(0, XMAX), ylim=(0, 1), yticks=[])
    ax.tick_params(axis='x', bottom=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

def ar6_world_band(ax):
    for j, category in enumerate(AR6_COLOR):
        colour = AR6_COLOR[category]
        ax.boxplot([ar6_values[category]], positions=[4 - j], vert=False, widths=.5, whis=(5, 95),
                   patch_artist=True, manage_ticks=False,
                   boxprops={'facecolor': colour, 'edgecolor': colour, 'linewidth': .6},
                   medianprops={'color': 'white', 'linewidth': 1.}, whiskerprops={'color': colour, 'linewidth': .6},
                   capprops={'color': colour, 'linewidth': .6},
                   flierprops={'marker': '.', 'markersize': 1.8, 'markerfacecolor': colour,
                               'markeredgecolor': colour, 'clip_on': False})
    ax.set(xlim=(0, XMAX), ylim=(.35, 4.65), yticks=[4, 3, 2, 1], yticklabels=list(AR6_COLOR))
    ax.tick_params(axis='y', length=0, pad=3)
    ax.tick_params(axis='x', bottom=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

def jurisdiction_panel(ax, country, model, show_ticks):
    benchmark = ar6_regional.loc[ar6_regional.study_jurisdiction.eq(country), 'rate_gt_yr'].to_numpy(float)
    if len(benchmark):
        lo, q1, q3, hi = np.quantile(benchmark, [.05, .25, .75, .95])
        ax.axvspan(q1, q3, color='#c6dbef', alpha=.65, lw=0, zorder=0)
        for value in (lo, hi):
            ax.axvline(value, color='#2171b5', lw=.6, zorder=1)
    marker, style = ('o', '-') if model == 'Logistic' else ('s', (0, (1.5, 1.0)))
    for k, s in enumerate(SCENARIOS):
        colour = RATE_COLOR[s]
        for stage, shift in [('pre', -.25), ('post', .25)]:
            q = intervals.loc[(s, country, model, stage)]
            if not q.available:
                ax.text(.98, k + shift, 'NA', transform=ax.get_yaxis_transform(), ha='right', va='center',
                        fontsize=6, color='#8b3434')
                continue
            if not q.fixed_target:
                ax.plot([q.p05, q.p95], [k + shift] * 2, color=colour, lw=.55, ls=style, zorder=3)
                ax.plot([q.p25, q.p75], [k + shift] * 2, color=colour, lw=1.35, ls=style, zorder=4)
            ax.plot(q.p50, k + shift, marker=marker, ms=3.2, ls='none', mfc='white' if stage == 'pre' else colour,
                    mec=colour, mew=.65, zorder=6)
    ax.set_xscale('symlog', linthresh=.001, linscale=1.)
    ax.set(xlim=LIMITS, ylim=(5.6, -.6))
    ax.set_xticks([0, .001, .01, .1, 1, 10])
    ax.set_xticklabels(['0', '10⁻³', '10⁻²', '10⁻¹', '10⁰', '10¹'])
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_yticks(range(6))
    ax.set_yticklabels([])
    ax.tick_params(axis='y', length=0, pad=3, labelsize=6)
    ax.tick_params(axis='x', length=2, pad=2, labelsize=6, labelbottom=show_ticks)
    for y in np.arange(.5, 5.5):
        ax.axhline(y, color='#ededed', lw=.35, zorder=0)
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.spines['bottom'].set_color('#999999')
    ax.spines['bottom'].set_linewidth(.45)
    if model == 'Logistic':
        ax.set_title({'US': 'United States', 'UK': 'United Kingdom', 'EU': 'EU group'}.get(country, country),
                     fontsize=7, loc='center', pad=5)

In [ ]:
def rate_figure():
    width, height = 183., 233.
    fig = plt.figure(figsize=(width / 25.4, height / 25.4))
    fy = lambda top: 1 - (top - 7.) / height
    def axis(x, top, w, h):
        return fig.add_axes([x / width, fy(top + h), w / width, h / height])
    def text(x, top, s, **kw):
        return fig.text(x / width, fy(top), s, va='top', **kw)
    def line(xs, ys, **kw):
        fig.add_artist(Line2D([x / width for x in xs], ys, transform=fig.transFigure, **kw))
    def rect(x, y, w, h, **kw):
        fig.add_artist(Rectangle((x / width, y), w / width, h, transform=fig.transFigure, **kw))

    text(5, 11, 'A', fontsize=8, fontweight='bold')
    text(18, 11, 'Included-jurisdiction study aggregate; AR6 benchmark: World', fontsize=6, color='#555555')
    bands = [axis(18, top, 160, h) for top, h in [(17, 20), (38, 4), (43, 12), (56, 4), (61, 20)]]
    histogram_band(bands[0], 'pre')
    density_band(bands[1], 'pre')
    density_band(bands[3], 'post')
    histogram_band(bands[4], 'post')
    ar6_world_band(bands[2])

    text(5, 91, 'B', fontsize=8, fontweight='bold')
    text(18, 91, COVERAGE_NOTE, fontsize=6, color='#555555')
    countries = ['China', 'US', 'Middle East', 'EU', 'Brazil', 'Canada', 'Indonesia', 'Australia', 'UK', 'Thailand']
    for group in range(2):
        for j, model in enumerate(MODELS):
            top = 101 + group * 46 + j * 22
            fig.text(8 / width, fy(top + 9.25), model, fontsize=7, color='.15', rotation=90,
                     rotation_mode='anchor', ha='center', va='center')
            for col, country in enumerate(countries[group * 5:(group + 1) * 5]):
                jurisdiction_panel(axis(18 + col * 32.5, top, 30, 18.5), country, model,
                                   show_ticks=(group == 1 and j == 1))
    text(98, 195, '2050 annual injection rate (Gt yr⁻¹)', fontsize=7, ha='center')

    rect(5, fy(237), 173, 36 / height, facecolor='white', edgecolor='black', lw=.65, zorder=0)
    text(8, 203, 'Modelled scenarios', fontsize=7)
    for j, s in enumerate(SCENARIOS):
        rect(8 + j * 28, fy(210), 3, 2 / height, facecolor=RATE_COLOR[s], edgecolor='none')
        text(12.5 + j * 28, 208, SCEN_LABEL[s], fontsize=6)
    for x, label, ls, marker in [(8, 'Logistic', '-', 'o'), (33, 'Gompertz', (0, (3, 1.5)), 's')]:
        line([x, x + 4], [fy(214)] * 2, color='.25', ls=ls, lw=.8, marker=marker, ms=2.6, markevery=[0])
        text(x + 6, 213, label, fontsize=6)
    for x, label, fill in [(61, 'Before (upper)', 'white'), (96, 'After (lower)', '.25')]:
        line([x], [fy(214)], color='.25', marker='o', mfc=fill, ls='none', ms=3, mew=.65)
        text(x + 3, 213, label, fontsize=6)
    line([131, 141], [fy(214)] * 2, color='.25', lw=.55)
    line([134, 138], [fy(214)] * 2, color='.25', lw=1.35)
    line([136], [fy(214)], color='.25', marker='o', mfc='.25', ls='none', ms=3, mew=.65)
    text(143, 213, 'Median, 25–75%, 5–95%', fontsize=6)
    text(8, 218, 'AR6 pathways', fontsize=7)
    line([43, 47], [fy(219)] * 2, color='#2171b5', lw=.6)
    text(49, 218, '5–95%', fontsize=6)
    rect(72, fy(220), 4, 2 / height, facecolor='#c6dbef', edgecolor='none')
    text(78, 218, '25–75%', fontsize=6)
    ar6_labels = {'C1': 'Limit warming to 1.5 °C (>50% likelihood), no/limited overshoot',
                  'C2': 'Return to 1.5 °C (>50% likelihood), high overshoot',
                  'C3': 'Limit warming to 2 °C (>67% likelihood)',
                  'C4': 'Limit warming to 2 °C (>50% likelihood)'}
    for j, (category, label) in enumerate(ar6_labels.items()):
        top, colour = 222.5 + j * 3.4, AR6_COLOR[category]
        cy = fy(top + 1)
        line([9, 19], [cy] * 2, color=colour, lw=.6)
        rect(12, cy - .75 / height, 4, 1.5 / height, facecolor=colour, edgecolor=colour, lw=.5)
        for x in [9, 19]:
            line([x, x], [cy - .65 / height, cy + .65 / height], color=colour, lw=.6)
        line([14], [cy], marker='o', ms=1.6, mfc='white', mec='white', mew=.1, ls='none')
        text(22, top, category, fontsize=6, fontweight='bold')
        text(30, top, label, fontsize=6)
        text(175, top, f'[{len(ar6_values[category])} model outcomes]', fontsize=6, ha='right')
    return fig

with mpl.rc_context({**BASE_RC, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6}):
    save(rate_figure(), 'fig4_rate2050_ar6')

## Figure 5. Allocation and reconstruction example

In [ ]:
K = 1.15
CASE = dict(country='UK', scenario='reference', allocation_order='descend')
SHORTFALL_COLOR, RECON_COLOR = '#B44A4D', '#D55E00'
FIG5_RC = {'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans'], 'pdf.fonttype': 42,
           'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6 * K, 'xtick.major.width': .8 * K, 'ytick.major.width': .8 * K,
           'xtick.labelsize': 6 * K, 'ytick.labelsize': 6 * K}
TITLE_FONT = ['Arial', 'DejaVu Sans']

allocation = read_csv(OUT / 'basin_allocation.csv')
curves = read_csv(OUT / 'reconstruction_paths.csv', usecols=KEYS + ['allocation_order', 'curve_kind', 'year', 'rate_mt_yr'])
select = lambda d: d[d.country.eq(CASE['country']) & d.scenario.eq(CASE['scenario'])]
uk_basins = allocation[allocation.country.eq(CASE['country'])].drop_duplicates('Region no').sort_values('Region no')
palette = list(mpl.colormaps['tab20'].colors) + list(mpl.colormaps['Set3'].colors)
BASIN_COLOR = dict(zip(uk_basins['Region no'], palette))
BASIN_NAME = dict(zip(uk_basins['Region no'], uk_basins['Region name']))

def curve_params(row, kind):
    if kind == 'original':
        return row.C_scenario_gt, row.r_original, row.tp_original
    return row.C_recon_order_gt, row.r_recon_order, row.tp_recon_order

def analytic_rate(row, kind, t):
    C, r, _ = curve_params(row, kind)
    dt = np.asarray(t, float) - 2030.
    if row.model == 'Logistic':
        z = (C / row.S0_gt - 1.) * np.exp(-r * dt)
        return C * r * z / (1. + z) ** 2
    z = np.log(C / row.S0_gt) * np.exp(-r * dt)
    return C * r * z * np.exp(-z)

def analytic_peak(row, kind):
    C, r, year = curve_params(row, kind)
    return year, C * r / (4. if row.model == 'Logistic' else np.e)

def rate_axes(ax, ymax):
    ax.set(xlim=(2030, 2180), ylim=(0, ymax * 1.12), xticks=[2030, 2050, 2100, 2180])
    ax.set_xlabel('Year', fontsize=7 * K, labelpad=2 * K)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=3))
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.2f}'))
    ax.tick_params(length=2.5 * K, pad=2 * K)
    ax.grid(axis='y', color='.93', lw=.4 * K)
    ax.set_axisbelow(True)

def mark_peak(ax, row, kind, color):
    year, peak = analytic_peak(row, kind)
    ax.vlines(year, 0, peak, color=color, ls=':', lw=.7 * K, alpha=.85)
    ax.plot(year, peak, marker='*', ms=6 * K, color=color, mec='white', mew=.35 * K, zorder=7)

def allocation_panel(ax, row, steps, ymax):
    curve = select(curves)[lambda d: d.model.eq(row.model) & d.allocation_order.eq(row.allocation_order)
                           & d.curve_kind.eq('original')].sort_values('year')
    years, demand = curve.year.to_numpy(float), curve.rate_mt_yr.to_numpy(float) / 1000.
    levels = steps['Step rate cumulative [Mt/y]'].to_numpy(float) / 1000.
    nodes = [years]
    for level in levels:
        cross = (demand[:-1] - level) * (demand[1:] - level) < 0
        nodes.append(years[:-1][cross] + (level - demand[:-1][cross]) / (demand[1:][cross] - demand[:-1][cross]))
    x = np.unique(np.concatenate(nodes))
    d = np.interp(x, years, demand)
    top, bottom = np.minimum(d, levels[:, None]), np.minimum(d, np.r_[0., levels[:-1]][:, None])
    for i, basin in enumerate(steps['Region no']):
        ax.fill_between(x, bottom[i], top[i], color=BASIN_COLOR[basin], lw=.2 * K, edgecolor='white')
    ax.fill_between(x, np.minimum(d, levels[-1]), d, color=SHORTFALL_COLOR, alpha=.22, lw=0)
    ax.plot(x, d, color='#222222', lw=1.1 * K)
    ax.axhline(levels[-1], color='#555555', ls=(0, (3, 2)), lw=.7 * K)
    ax.plot(2050, row.rate_2050_original, marker='o', ms=3 * K, mfc='white', mec='#222222', mew=.7 * K, zorder=5)
    rate_axes(ax, ymax)
    mark_peak(ax, row, 'original', '#222222')
    ax.set_ylabel('Injection rate (Gt/yr)', fontsize=7 * K, labelpad=3 * K)

def reconstruction_panel(ax, row, ymax):
    x = np.unique(np.r_[np.linspace(2030, 2180, 1500), row.tp_original, row.tp_recon_order])
    rate_axes(ax, ymax)
    for kind, colour, style in [('original', '.5', '--'), ('reconstructed', RECON_COLOR, '-')]:
        ax.plot(x, analytic_rate(row, kind, x), color=colour, ls=style, lw=1. * K)
        mark_peak(ax, row, kind, colour)
    ax.axhline(row.rescreen_final_allocated_rate_mt_yr / 1000., color='#555555', ls=(0, (3, 2)), lw=.7 * K)
    ax.plot(2050, row.rate_2050_original, marker='o', ms=3 * K, mfc='white', mec='#222222', mew=.7 * K, zorder=8)
    ax.tick_params(labelleft=False)

def allocation_example_figure():
    W, H, legend_y = 180., 113.4, 110.2
    fig = plt.figure(figsize=(W / 25.4, H / 25.4), dpi=72)
    text = lambda x, y, s, size, bold=False: fig.text(x / W, 1 - y / H, s, family=TITLE_FONT, fontsize=size,
                                                       fontweight='bold' if bold else 'normal', va='baseline')
    cases = select(reconstruction)
    used = set()
    for model, row_top, letters in [('Logistic', 5.2, 'AB'), ('Gompertz', 61.2, 'CD')]:
        group = cases[cases.model.eq(model)]
        row = group[group.allocation_order.eq(CASE['allocation_order'])].iloc[0]
        ymax = max(max(analytic_peak(r, 'original')[1], analytic_peak(r, 'reconstructed')[1],
                       r.rescreen_final_allocated_rate_mt_yr / 1000.) for _, r in group.iterrows())
        steps = select(allocation)[lambda d: d.model.eq(model) & d.allocation_order.eq(row.allocation_order)].sort_values('Step')
        used |= set(steps['Region no'])
        for left, draw in [(1 + 12 * K, lambda ax: allocation_panel(ax, row, steps, ymax)),
                           (104 + 2 * K, lambda ax: reconstruction_panel(ax, row, ymax))]:
            draw(fig.add_axes([left / W, 1 - (row_top + 29 * K) / H, 53 * K / W, 28 * K / H]))
        text(2, row_top - 1.8, f'{letters[0]}   {model}: basin allocation', 8.5, True)
        text(104, row_top - 1.8, f'{letters[1]}   {model}: reconstruction', 8.5, True)
    basins = [b for b in BASIN_NAME if b in used]
    labels = [text(0, legend_y - .25, BASIN_NAME[b], 7.5) for b in basins]
    renderer = fig.canvas.get_renderer()
    widths = [4.0 + label.get_window_extent(renderer).width / fig.dpi * 25.4 for label in labels]
    gap = (176 - sum(widths)) / (len(basins) - 1)
    x = 2.0
    for basin, label, w in zip(basins, labels, widths):
        fig.add_artist(Rectangle((x / W, 1 - legend_y / H), 2.5 / W, 2.5 / H, transform=fig.transFigure,
                                 facecolor=BASIN_COLOR[basin], edgecolor='none'))
        label.set_x((x + 4.0) / W)
        x += w + gap
    return fig

with mpl.rc_context(FIG5_RC):
    save(allocation_example_figure(), 'fig5_allocation_example')